<a href="https://colab.research.google.com/github/ksuaray/M4DS/blob/MATH-170-Spring-2026/Lab5_Perceptrons_Activations_Reg_Class.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MATH 170 — Lab 5  
# Perceptrons, Derivatives, and Activation Functions (1 input → 1 output)

**Big Idea:** A perceptron with one input is just a function with one parameter:

$$
\hat{y}=\phi(wx)
$$

- $x$: input feature  
- $w$: weight (the only parameter today)  
- $\phi$: activation function  

Training = choosing $w$ so that a **loss function** $E(w)$ is as small as possible.

---

## How to work through this lab (self-guided)
In each section you’ll follow this routine:

1) **Try a few values of $w$** (data science style)  
2) **Compute the derivative $E'(w)$** (calculus style)  
3) Use the **increasing–decreasing test** + **critical numbers** to explain what’s happening  

> When you see **GTW**, you should write code or write a short response.

---

## Learning Targets (VARIOSA)
- **V:** plot models + loss curves + activation functions  
- **A:** approximate “best” $w$ using tables / grids  
- **I/O:** tune parameters by minimizing $E(w)$  
- **S:** compute derivatives with SymPy  
- **R:** interpret logistic outputs as probabilities for classification


In [ ]:
# Part 0 — Setup (run this first)
import numpy as np
import sympy as sp
import pandas as pd
import plotly.express as px

sp.init_printing()

# Symbols
x = sp.Symbol('x', real=True)
w = sp.Symbol('w', real=True)

print("Ready!")



---

# Part 1 — Regression Perceptron (Identity Activation)

## Context: TikTok Before Noon vs Energy at 2PM
We track:
- $x$ = hours on TikTok before noon  
- $y$ = energy level at 2PM (1–10 scale)

We fit a simple perceptron regression model (identity activation $\phi(z)=z$):

$$
\hat{y} = wx
$$

### Dataset $n=5$


In [ ]:
# Data (n=5)
x_data = np.array([1, 2, 3, 4, 5], dtype=float)
y_data = np.array([3, 4, 6, 8, 7], dtype=float)

df_reg = pd.DataFrame({"x": x_data, "y": y_data})
df_reg



### 1.1 Visualize the data

In [ ]:
fig = px.scatter(df_reg, x="x", y="y", title="TikTok hours (x) vs Energy at 2PM (y)")
fig.show()


### **GTW:** Does it look like energy increases or decreases as TikTok hours increase?



## 1.2 Build the loss function $E(w)$

Model:
$$\hat{y}_i = wx_i$$

Least squares loss:
$$E(w)=\sum_{i=1}^{n}(y_i-wx_i)^2$$

In [ ]:
# Build E(w) symbolically with SymPy
E = 0
for xi, yi in zip(x_data, y_data):
    E += (w*xi - yi)**2

E = sp.simplify(E)
E


### **GTW:** What do you think the `+=` in the code cell above is doing?



## 1.3 Try different values of $w$ AND check the derivative

When you “try weights” you’re really probing whether the loss is increasing or decreasing.




### **GTW:** State the increasing decreasing test below by filling in the ... :


- If $E'(w) < 0$, then $E(w)$ is **...** at that $w$.
- If $E'(w) > 0$, then $E(w)$ is **...** at that $w$.

In the table below, we'll compute both $E(w)$ and $E'(w)$ for each $w$.


In [ ]:
# Compute the derivative
E_prime = sp.simplify(sp.diff(E, w))
E_prime


In [ ]:
# Numeric versions for quick evaluation
E_num = sp.lambdify(w, E, "numpy")
Eprime_num = sp.lambdify(w, E_prime, "numpy")

# GTW: Try these weights (you can add more)
w_try = np.array([-3, -2, -1, 0, 1, 2, 3], dtype=float)

table = pd.DataFrame({
    "w": w_try,
    "E(w)": E_num(w_try),
    "E'(w)": Eprime_num(w_try)
})

# Add an "increasing/decreasing" interpretation
table["Behavior near w"] = np.where(table["E'(w)"] > 0, "increasing", np.where(table["E'(w)"] < 0, "decreasing", "flat (critical)"))
table


### **GTW:** Replace the ... below with a more narrow range of $x$-values to identify the minimum value of the $E(w)$ correct to one decimal place. This means that $\Delta x = 0.01$. In the table above, $\Delta x = 1$ since the difference between consecutive $x$ values is 1. Use six $x$ values.

In [ ]:
# Numeric versions for quick evaluation
E_num = sp.lambdify(w, E, "numpy")
Eprime_num = sp.lambdify(w, E_prime, "numpy")


w_try = np.array([...], dtype=float)        # GTW: Replace the x values

table = pd.DataFrame({
    "w": w_try,
    "E(w)": E_num(w_try),
    "E'(w)": Eprime_num(w_try)
})

# Add an "increasing/decreasing" interpretation
table["Behavior near w"] = np.where(table["E'(w)"] > 0, "increasing", np.where(table["E'(w)"] < 0, "decreasing", "flat (critical)"))
table


## 1.4 Loss curve + where the minimum happens

### **GTW:** Run the code below to plot $E(w)$ for a wide range of $w$. Where does the minimum *look* like it occurs?

In [ ]:
w_vals = np.linspace(-5, 7, 600)
loss_vals = E_num(w_vals)

df_loss = pd.DataFrame({"w": w_vals, "E(w)": loss_vals})
fig = px.line(df_loss, x="w", y="E(w)", title="Loss curve E(w) for regression perceptron")
fig.show()



## 1.5 Critical number and First Derivative Test (calculus)

A **critical number** occurs where:

$$
E'(w)=0 \quad \text{or} \quad E'(w) \text{ does not exist}
$$

Here $E'(w)$ exists everywhere (it’s a polynomial), so we solve:

$$
E'(w)=0
$$

### **GTW:** Solve for the critical number $w^*$. Then explain why it should be a minimum using an increasing/decreasing argument.


In [ ]:
w_star = sp.solve(sp.Eq(E_prime, 0), w)
w_best = float(w_star[0])
w_best

In [ ]:
# Plot data + best-fit model yhat = w_best * x
xs = np.linspace(0.5, 5.5, 200)
yhat = w_best * xs

fig = px.scatter(df_reg, x="x", y="y", title=f"Best regression perceptron: yhat = {w_best:.3f} x")
fig = fig.add_scatter(x=xs, y=yhat, mode="lines", name="model")
fig.show()



### **GTW (short response):** At the best weight $w^*$, what is $E'(w^*)$?  What does that tell you about the slope of the loss curve at the minimum?


In [ ]:
Eprime_num(...)



---

# Part 2 — Binary Classification (Same Architecture, different $\phi$)


## Context: TikTok Before Noon vs Headache at 2PM

Now outputs are classes (categories). For example with the same feature above, we could assign the corresponding label based on the question: "Does amount of time on TikTok predict whether or not users suffer headaches later the same day?" Let
- 0 = No headache
- 1 = Headache

This time let's use the score:
$$
z = wx - 0.4
$$

Here the $-0.4$ is referred to as the $bias$. Also, we'll try two activation functions. We will consider $H(z)$ and $\sigma(z)$

---

## Part 2A — Heaviside Step Activation (hard threshold)

$$
H(z)=
\begin{cases}
0 & z<0\\
1 & z\ge 0
\end{cases}
$$

Model:
$$
\hat{y}=H(wx-0.4)
$$

### ***GTW:*** Why do you think $H(z)$ is a good function to use for binary classification?

In [ ]:
# Classification dataset (n=5)
x_cls = np.array([1, 2, 3, 4, 5], dtype=float)
y_cls = np.array([0, 1, 0, 0, 1], dtype=int)

df_cls = pd.DataFrame({"x": x_cls, "y": y_cls})
df_cls



### 2A.1 Try a few weights and count misclassifications

1. Try $w = -0.1,0.1,0.2,1,2$.
2. For each $w$, compute predictions $\hat{y}$.
3. Count how many are wrong.

> Notice: With Heaviside, the model output can “jump” suddenly.


In [ ]:
def heaviside(z):
    return (z >= 0).astype(int)

def misclassifications(y_true, y_pred):
    return int(np.sum(y_true != y_pred))

w_try_step = np.array([-0.1,0.1,0.2,1,2], dtype=float)

rows = []
for wv in w_try_step:
    yhat = heaviside(wv * x_cls-0.4)
    rows.append({
        "w": wv,
        "predictions": list(yhat),
        "misclassifications": misclassifications(y_cls, yhat)
    })

pd.DataFrame(rows)



### 2A.2 Why calculus struggles here
The step function is **not differentiable** at 0, so it doesn’t give us a useful derivative-based “direction” for improving $w$.



## Part 2B — Logistic Activation (soft threshold)

$$
\sigma(z)=\frac{1}{1+e^{-z}}
$$

Model:
$$
\hat{y}=\sigma(wx)
$$

Now $\hat{y}$ is a number between 0 and 1, which we interpret as a **probability of class 1**.

To convert probability to a class label, a common rule is:
- predict 1 if $\hat{y}\ge 0.5$
- predict 0 otherwise

### Loss function for training (squared error)
$$
E(w)=\sum_{i=1}^{n}\left(y_i-\sigma(wx_i)\right)^2
$$

Key difference: **This loss is differentiable**, so we can study $E'(w)$.


In [ ]:
# Build logistic predictions symbolically (as a function of x, w)
sigma_sym = 1/(1 + sp.exp(-(w*x)))
sigma_sym



### 2B.1 Try different $w$ values AND check $E'(w)$ each time

**GTW:**
Try:
$$
w=-1,-0.5, 0, 0.5, 1
$$

For each $w$:
- compute $E(w)$
- compute $E'(w)$
- decide if the loss is increasing or decreasing there


In [ ]:
Elog_num = sp.lambdify(w, E_log, "numpy")
Elogprime_num = sp.lambdify(w, E_log_prime, "numpy")

w_try_log = np.array([-1,-0.5, 0, 0.5, 1], dtype=float)

tab_log = pd.DataFrame({
    "w": w_try_log,
    "E(w)": Elog_num(w_try_log),
    "E'(w)": Elogprime_num(w_try_log),
})
tab_log["Behavior near w"] = np.where(tab_log["E'(w)"] > 0, "increasing", np.where(tab_log["E'(w)"] < 0, "decreasing", "flat (critical)"))
tab_log


### **GTW:** Replace the ... below with a more narrow range of $x$-values to identify the minimum value of the $E(w)$ correct to one decimal place. This means that $\Delta x = 0.01$. In the table above, $\Delta x = 0.5$ since the difference between consecutive $x$ values is 0.5. Use six $x$ values.

In [ ]:
Elog_num = sp.lambdify(w, E_log, "numpy")
Elogprime_num = sp.lambdify(w, E_log_prime, "numpy")

w_try_log = np.array([...], dtype=float)

tab_log = pd.DataFrame({
    "w": w_try_log,
    "E(w)": Elog_num(w_try_log),
    "E'(w)": Elogprime_num(w_try_log),
})
tab_log["Behavior near w"] = np.where(tab_log["E'(w)"] > 0, "increasing", np.where(tab_log["E'(w)"] < 0, "decreasing", "flat (critical)"))
tab_log


### 2B.2 Visualize: predictions for different weights

### **GTW:** What happens to the direction and “sharpness” of the probability curve as $w$ increases? How is this better than the Heaviside function"


In [ ]:
def logistic(z):
    return 1/(1+np.exp(-z))

xs = np.linspace(-4, 4, 600)

curves = []
for wv in [-1,-0.5, 0, 0.5, 1]:
    curves.append(pd.DataFrame({
        "x": xs,
        "yhat": logistic(wv*xs),
        "w": f"w={wv}"
    }))

df_curves = pd.concat(curves, ignore_index=True)
fig = px.line(df_curves, x="x", y="yhat", color="w",
              title="Logistic activation outputs: yhat = sigma(w x)")
fig.show()



### 2B.3 Loss curve + (approx) critical number

**GTW:**
1. Plot $E(w)$ vs $w$ for $w\in[-2,6]$.
2. Where is the minimum?
3. Use the sign of $E'(w)$ to explain why the minimum should be near that point.


In [ ]:
w_vals = np.linspace(-2, 6, 800)
df_elog = pd.DataFrame({"w": w_vals, "E(w)": Elog_num(w_vals)})

fig = px.line(df_elog, x="w", y="E(w)", title="Logistic classification loss E(w)")
fig.show()


In [ ]:
# Approximate the minimizer by grid search (data science style)
w_grid = np.linspace(-2, 6, 4001)
E_grid = Elog_num(w_grid)
idx = np.argmin(E_grid)
w_best_log = float(w_grid[idx])
E_best_log = float(E_grid[idx])

w_best_log, E_best_log


In [ ]:
# Check derivative at the best weight (should be near 0)
Elogprime_num(...)



### **GTW (short response):**
- If $E'(w)$ is negative at your current $w$, should you increase or decrease $w$?
- If $E'(w)$ is positive, should you increase or decrease $w$?
Explain using the increasing–decreasing test.
